# 03 · Gold Build

Builds two Gold Delta tables from `asset_mgmt.silver.daily_prices`.
All columns carry `COMMENT` strings for Databricks Genie.

| Table | Grain | Purpose |
|---|---|---|
| `gold.dashboard_latest_prices` | One row per ticker | Latest snapshot + short-term return metrics |
| `gold.ticker_performance` | One row per ticker per date | Full history with rolling analytics |

Both tables use `MERGE INTO` — safe to re-run after every Silver update.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType

SILVER_TABLE     = "asset_mgmt.silver.daily_prices"
GOLD_DASHBOARD   = "asset_mgmt.gold.dashboard_latest_prices"
GOLD_PERFORMANCE = "asset_mgmt.gold.ticker_performance"

print(f"Silver : {SILVER_TABLE}")
print(f"Gold 1 : {GOLD_DASHBOARD}")
print(f"Gold 2 : {GOLD_PERFORMANCE}")

In [ ]:
# ── DDL: gold.dashboard_latest_prices ────────────────────────────────────
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_DASHBOARD} (
    Ticker          STRING    COMMENT 'Stock ticker symbol (e.g. AAPL)',
    Sector          STRING    COMMENT 'GICS sector classification (Tech, Finance, Healthcare, Energy, Consumer, Benchmark)',
    latest_date     DATE      COMMENT 'Most recent trading date with available data',
    open            DOUBLE    COMMENT 'Opening price on latest_date in USD',
    high            DOUBLE    COMMENT 'Intraday high price on latest_date in USD',
    low             DOUBLE    COMMENT 'Intraday low price on latest_date in USD',
    close           DOUBLE    COMMENT 'Adjusted closing price on latest_date in USD',
    volume          LONG      COMMENT 'Number of shares traded on latest_date',
    daily_return    DOUBLE    COMMENT 'Day-over-day return on latest_date: (close - prev_close) / prev_close. Null for first trading day.',
    weekly_return   DOUBLE    COMMENT '5-trading-day return: (close - close_5d_ago) / close_5d_ago. Null if fewer than 5 prior days exist.',
    monthly_return  DOUBLE    COMMENT '21-trading-day return: (close - close_21d_ago) / close_21d_ago. Null if fewer than 21 prior days exist.',
    _source_type    STRING    COMMENT 'Ingestion source of the latest row: seed | batch_2 | live',
    _ingest_timestamp TIMESTAMP COMMENT 'When the latest row was ingested into Bronze'
)
USING DELTA
COMMENT 'Gold table: one row per ticker showing the latest price snapshot and short-term return metrics. Refreshed on every pipeline run.'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true',
    'project'                          = 'EoDA Assignment 2 - NOVA SBE',
    'layer'                            = 'gold',
    'grain'                            = 'one row per ticker'
)
""")
print(f"Table ready: {GOLD_DASHBOARD}")

In [ ]:
# ── DDL: gold.ticker_performance ─────────────────────────────────────────
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_PERFORMANCE} (
    Ticker                 STRING    COMMENT 'Stock ticker symbol (e.g. AAPL)',
    Date                   DATE      COMMENT 'Trading date',
    Sector                 STRING    COMMENT 'GICS sector classification (Tech, Finance, Healthcare, Energy, Consumer, Benchmark)',
    open                   DOUBLE    COMMENT 'Opening price in USD',
    high                   DOUBLE    COMMENT 'Intraday high price in USD',
    low                    DOUBLE    COMMENT 'Intraday low price in USD',
    close                  DOUBLE    COMMENT 'Adjusted closing price in USD',
    volume                 LONG      COMMENT 'Number of shares traded',
    daily_return           DOUBLE    COMMENT 'Day-over-day return: (close - prev_close) / prev_close. Null for first trading day per ticker.',
    cumulative_return      DOUBLE    COMMENT 'Total return since first available date: (close - first_close) / first_close',
    rolling_30d_avg_close  DOUBLE    COMMENT '30-trading-day rolling average of the closing price in USD',
    rolling_30d_volatility DOUBLE    COMMENT '30-trading-day rolling standard deviation of daily_return (daily, not annualised)',
    volume_flag            BOOLEAN   COMMENT 'True if Volume is negative — signals a data quality issue upstream',
    _source_type           STRING    COMMENT 'Ingestion source of this row: seed | batch_2 | live'
)
USING DELTA
COMMENT 'Gold table: one row per ticker per trading date with full OHLCV history and rolling analytics. Use this table for time-series and cross-ticker queries.'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true',
    'project'                          = 'EoDA Assignment 2 - NOVA SBE',
    'layer'                            = 'gold',
    'grain'                            = 'one row per ticker per date'
)
""")
print(f"Table ready: {GOLD_PERFORMANCE}")

In [ ]:
# ── Read Silver and compute all enrichments in one pass ───────────────────
silver = spark.table(SILVER_TABLE)
print(f"Silver rows : {silver.count():,}")

# Window definitions
w_ticker       = Window.partitionBy("Ticker").orderBy("Date")
w_ticker_30d   = Window.partitionBy("Ticker").orderBy("Date").rowsBetween(-29, 0)

enriched = (
    silver
    # Lags for short-term return metrics (used in dashboard table)
    .withColumn("close_5d_ago",  F.lag("Close", 5).over(w_ticker))
    .withColumn("close_21d_ago", F.lag("Close", 21).over(w_ticker))
    .withColumn("weekly_return",
        F.when(
            F.col("close_5d_ago").isNotNull() & (F.col("close_5d_ago") != 0),
            (F.col("Close") - F.col("close_5d_ago")) / F.col("close_5d_ago")
        ).otherwise(F.lit(None).cast(DoubleType()))
    )
    .withColumn("monthly_return",
        F.when(
            F.col("close_21d_ago").isNotNull() & (F.col("close_21d_ago") != 0),
            (F.col("Close") - F.col("close_21d_ago")) / F.col("close_21d_ago")
        ).otherwise(F.lit(None).cast(DoubleType()))
    )
    # Cumulative return from each ticker's first date (used in performance table)
    .withColumn("first_close", F.first("Close").over(w_ticker))
    .withColumn("cumulative_return",
        F.when(
            F.col("first_close").isNotNull() & (F.col("first_close") != 0),
            (F.col("Close") - F.col("first_close")) / F.col("first_close")
        ).otherwise(F.lit(None).cast(DoubleType()))
    )
    # 30-day rolling metrics (used in performance table)
    .withColumn("rolling_30d_avg_close",  F.avg("Close").over(w_ticker_30d))
    .withColumn("rolling_30d_volatility", F.stddev("daily_return").over(w_ticker_30d))
    # Row number for picking latest row per ticker
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("Ticker").orderBy(F.col("Date").desc())
    ))
    .drop("close_5d_ago", "close_21d_ago", "first_close")
)

print("Enrichment complete.")

In [ ]:
# ── Stage: gold.dashboard_latest_prices ──────────────────────────────────
dashboard_staged = (
    enriched
    .filter(F.col("rn") == 1)   # keep only the most recent row per ticker
    .select(
        "Ticker",
        "Sector",
        F.col("Date").alias("latest_date"),
        F.col("Open").alias("open"),
        F.col("High").alias("high"),
        F.col("Low").alias("low"),
        F.col("Close").alias("close"),
        F.col("Volume").alias("volume"),
        "daily_return",
        "weekly_return",
        "monthly_return",
        "_source_type",
        "_ingest_timestamp",
    )
)

dashboard_staged.createOrReplaceTempView("dashboard_staged")
print(f"Dashboard rows : {dashboard_staged.count()}  (should equal number of tickers)")
display(dashboard_staged.orderBy("Sector", "Ticker"))

In [ ]:
# ── MERGE INTO gold.dashboard_latest_prices ───────────────────────────────
spark.sql(f"""
MERGE INTO {GOLD_DASHBOARD} AS target
USING dashboard_staged AS source
  ON target.Ticker = source.Ticker
WHEN MATCHED AND source._ingest_timestamp > target._ingest_timestamp
  THEN UPDATE SET *
WHEN NOT MATCHED
  THEN INSERT *
""")

print("MERGE complete — gold.dashboard_latest_prices")
print(f"Rows now : {spark.table(GOLD_DASHBOARD).count()}")

In [ ]:
# ── Stage: gold.ticker_performance ───────────────────────────────────────
performance_staged = (
    enriched
    .select(
        "Ticker",
        "Date",
        "Sector",
        F.col("Open").alias("open"),
        F.col("High").alias("high"),
        F.col("Low").alias("low"),
        F.col("Close").alias("close"),
        F.col("Volume").alias("volume"),
        "daily_return",
        "cumulative_return",
        "rolling_30d_avg_close",
        "rolling_30d_volatility",
        "volume_flag",
        "_source_type",
    )
)

performance_staged.createOrReplaceTempView("performance_staged")
print(f"Performance rows : {performance_staged.count():,}")
display(performance_staged.limit(5))

In [ ]:
# ── MERGE INTO gold.ticker_performance ───────────────────────────────────
spark.sql(f"""
MERGE INTO {GOLD_PERFORMANCE} AS target
USING performance_staged AS source
  ON target.Ticker = source.Ticker
 AND target.Date   = source.Date
WHEN MATCHED
  THEN UPDATE SET *
WHEN NOT MATCHED
  THEN INSERT *
""")

print("MERGE complete — gold.ticker_performance")
print(f"Rows now : {spark.table(GOLD_PERFORMANCE).count():,}")

In [ ]:
# ── Verify both Gold tables ───────────────────────────────────────────────
print("=== dashboard_latest_prices ===")
display(
    spark.table(GOLD_DASHBOARD)
    .select("Ticker", "Sector", "latest_date", "close", "daily_return", "weekly_return", "monthly_return")
    .orderBy("Sector", "Ticker")
)

print("=== ticker_performance summary ===")
display(
    spark.table(GOLD_PERFORMANCE)
    .groupBy("Sector")
    .agg(
        F.countDistinct("Ticker").alias("tickers"),
        F.count("*").alias("rows"),
        F.min("Date").alias("date_min"),
        F.max("Date").alias("date_max"),
        F.round(F.avg("cumulative_return") * 100, 2).alias("avg_cumulative_return_pct"),
        F.round(F.avg("rolling_30d_volatility") * 100, 4).alias("avg_30d_volatility_pct"),
    )
    .orderBy("Sector")
)